# Keecas Quickstart Notebook

This notebook demonstrates the main patterns and features of keecas for symbolic math and units-aware calculations.

**Getting Started:**
1. Run the setup cell below to import keecas
2. Try the examples in each section
3. Modify values and re-run cells to see the results

**Keecas Conventions:**
- Use `_p` for cell-local parameters
- Use `_e` for cell-local expressions  
- Use `_v` for cell-local evaluated values
- Use `params` and `eqn` for notebook-global persistence

## Setup - Run This First

In [ ]:
# Essential keecas imports
from keecas import check, pc, show_eqn, symbols, u

# Configuration (uncomment to customize)
# config.display.katex = True              # For KaTeX compatibility
# config.display.print_label = True        # Print labels in dev mode
# config.latex.eq_prefix = "eq-example-"   # Custom label prefix
# config.language.language = "it"          # Language for translations

# Global persistence dictionaries
params = {}  # Global parameters that persist across cells
eqn = {}     # Global expressions that persist across cells

print("✅ Keecas setup complete! Ready for calculations.")

## Example 1: Basic Calculation

Simple stress calculation with units and symbolic math.

In [ ]:
# Define symbols (use LaTeX notation for better rendering)
F, A, sigma = symbols(r"F, A, \sigma")

# Parameters with units
_p = {
    F: 10 * u.kN,
    A: 50 * u.cm**2,
}

# Symbolic expressions
_e = {
    sigma: "F / A" | pc.parse_expr,
}

# Evaluation with unit conversion
_v = {
    k: v | pc.subs(_e | _p) | pc.convert_to([u.MPa]) | pc.N
    for k, v in _e.items()
}

# Display the results
show_eqn([_p | _e, _v])

## Example 2: Persistent Calculations

Using global dictionaries for calculations that span multiple cells.

In [ ]:
# Beam deflection setup - store in global params
q, L, E, I, delta = symbols(r"q, L, E, I, \delta")

_p = {
    q: 5 * u.kN/u.m,      # Distributed load
    L: 8 * u.m,           # Beam length
    E: 200 * u.GPa,       # Young's modulus
    I: 8540 * u.cm**4,    # Moment of inertia
}
params.update(_p)  # Save to global parameters

print(f"📚 Stored {len(_p)} parameters globally")
show_eqn([_p])

In [ ]:
# Deflection calculation - use global params
_e = {
    delta: "5 * q * L^4 / (384 * E * I)" | pc.parse_expr,
}
eqn.update(_e)  # Save to global expressions

# Evaluate using both local and global data
_v = {
    k: v | pc.subs(eqn | params) | pc.convert_to([u.mm]) | pc.N
    for k, v in _e.items()
}

show_eqn([_e, _v])
print(f"💾 Global storage: {len(params)} params, {len(eqn)} expressions")

## Example 3: Verification Checks

Using keecas `check()` function for engineering verifications.

In [ ]:
# Define limit state variables
sigma_Sd, sigma_Rd, tau_Sd, tau_Rd = symbols(r"\sigma_{Sd}, \sigma_{Rd}, \tau_{Sd}, \tau_{Rd}")

# Example loads and resistances
_p = {
    sigma_Sd: 180 * u.MPa,  # Design stress
    sigma_Rd: 235 * u.MPa,  # Resistance
    tau_Sd: 85 * u.MPa,     # Design shear stress
    tau_Rd: 135 * u.MPa,    # Shear resistance
}

# Verification expressions (utilization ratios)
_expr = [sigma_Sd / sigma_Rd, tau_Sd / tau_Rd]
_v = {k: k | pc.subs(_p) | pc.N for k in _expr}

# Verification checks (should be ≤ 1.0)
_c = {k: check(v, 1.0) for k, v in _v.items()}

show_eqn([_v, _c])

## Example 4: Advanced Features

Labels, descriptions, and custom formatting.

In [ ]:
# Complex engineering calculation with all features
P, f_c, A_c, f_y, A_s, N_Rd = symbols(r"P, f_c, A_c, f_y, A_s, N_{Rd}")

# Parameters
_p = {
    P: 1200 * u.kN,         # Applied load
    f_c: 25 * u.MPa,        # Concrete strength
    A_c: 0.25 * u.m**2,     # Concrete area
    f_y: 500 * u.MPa,       # Steel yield strength
    A_s: 2500 * u.mm**2,    # Steel area
}

# Expression
_e = {
    N_Rd: "0.85 * f_c * A_c + f_y * A_s" | pc.parse_expr,
}

# Evaluation
_v = {
    k: v | pc.subs(_e | _p) | pc.convert_to([u.kN]) | pc.N
    for k, v in _e.items()
}

# Descriptions
_d = {
    P: "Applied axial load",
    f_c: "Concrete compressive strength",
    A_c: "Concrete cross-sectional area",
    f_y: "Steel yield strength",
    A_s: "Steel reinforcement area",
    N_Rd: "Design resistance",
}

# Labels for cross-referencing
_l = {k: str(k).replace('\\', '').replace('{', '').replace('}', '') for k in _e.keys()}

# Verification
_c = {P: check(P | pc.subs(_p) | pc.N, _v[N_Rd])}

# Display with all features
show_eqn([_p | _e, _v, _d, _c],
         label=_l,
         float_format="{:.1f}")

## Your Playground

Use the cells below for your own calculations. Copy patterns from above or create your own!

In [ ]:
# Your calculation here



In [ ]:
# Another calculation



## Quick Reference

**Essential imports:**
```python
from keecas import symbols, u, pc, show_eqn, config, check
```

**Basic pattern:**
```python
# 1. Define symbols
a, b, result = symbols("a, b, result")

# 2. Parameters
_p = {a: 5*u.m, b: 10*u.m}

# 3. Expressions  
_e = {result: "a * b" | pc.parse_expr}

# 4. Evaluate
_v = {k: v | pc.subs(_e | _p) | pc.N for k, v in _e.items()}

# 5. Display
show_eqn([_p | _e, _v])
```

**Configuration:**
- `config.display.katex = True` - KaTeX compatibility
- `config.language.language = "it"` - Set language
- `config.latex.eq_prefix = "eq-"` - Equation label prefix

**More examples:** See `docs/CELL_TEMPLATE.py` for complete patterns.